In [1]:
import sys

!{sys.executable} -m pip install --no-index --no-deps --force-reinstall \
  /kaggle/input/datasets/unsseo/onnx-whl/onnxruntime-1.25.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl

Processing /kaggle/input/datasets/unsseo/onnx-whl/onnxruntime-1.25.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl


In [2]:
%env DEBUG_TRAIN_SOUNDSCAPE_LIMIT=16
%env PERCH_MAX_ROWS=0 #전체
#%env PERCH_TOP_WINDOWS_PER_FILE=1
%env PERCH_SELECT_BY=activity

%env BASELINE_MODEL_PATH=/kaggle/input/notebooks/antoinemasq/birdclef-2026-pytorch-baseline-training/models/8658fa9ee3393ad66210d15d4fd2063c41d50697e9fe201dbe9153e375ae2abf.pth
%env BASELINE_HISTORY_CSV=/kaggle/input/notebooks/antoinemasq/birdclef-2026-pytorch-baseline-training/history/8658fa9ee3393ad66210d15d4fd2063c41d50697e9fe201dbe9153e375ae2abf.csv
%env PERCH_PROBE_MODEL_PATHS=/kaggle/input/datasets/unsseo/perch-onnx/perch_feature_probe_full (2).pt
%env PERCH_PROBE_HISTORY_CSV=/kaggle/input/datasets/unsseo/perch-onnx/perch_feature_probe_history (1).csv
%env PERCH_MODEL_DIR=/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2/2
%env PERCH_USE_ONNX=1
%env PERCH_ONNX_PATH=/kaggle/input/datasets/unsseo/perch-onnx-backbone/perch_v2_backbone.onnx



env: DEBUG_TRAIN_SOUNDSCAPE_LIMIT=16
env: PERCH_MAX_ROWS=0 #전체
env: PERCH_SELECT_BY=activity
env: BASELINE_MODEL_PATH=/kaggle/input/notebooks/antoinemasq/birdclef-2026-pytorch-baseline-training/models/8658fa9ee3393ad66210d15d4fd2063c41d50697e9fe201dbe9153e375ae2abf.pth
env: BASELINE_HISTORY_CSV=/kaggle/input/notebooks/antoinemasq/birdclef-2026-pytorch-baseline-training/history/8658fa9ee3393ad66210d15d4fd2063c41d50697e9fe201dbe9153e375ae2abf.csv
env: PERCH_PROBE_MODEL_PATHS=/kaggle/input/datasets/unsseo/perch-onnx/perch_feature_probe_full (2).pt
env: PERCH_PROBE_HISTORY_CSV=/kaggle/input/datasets/unsseo/perch-onnx/perch_feature_probe_history (1).csv
env: PERCH_MODEL_DIR=/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2/2
env: PERCH_USE_ONNX=1
env: PERCH_ONNX_PATH=/kaggle/input/datasets/unsseo/perch-onnx-backbone/perch_v2_backbone.onnx


In [3]:
import ast
import glob
import json
import os
import random
from time import time

import numpy as np
import pandas as pd
import torch
import torchaudio
import torchvision
import torch.nn as nn


KAGGLE_PROJECT_DIR = "/kaggle/working"
KAGGLE_COMPETITION_DATA_DIR = "/kaggle/input/competitions/birdclef-2026"
PROJECT_DIR = KAGGLE_PROJECT_DIR if os.path.isdir(KAGGLE_PROJECT_DIR) else os.getcwd()


def resolve_default_data_path():
    candidates = [
        KAGGLE_COMPETITION_DATA_DIR,
        os.path.join("/kaggle/input", "birdclef-2026"),
        os.path.join(PROJECT_DIR, "birdclef-2026"),
    ]
    for path in candidates:
        if path and os.path.exists(os.path.join(path, "taxonomy.csv")):
            return path
    return os.path.join(PROJECT_DIR, "birdclef-2026")


DATA_PATH = os.environ.get("BIRDCLEF_DATA_PATH", resolve_default_data_path())
HISTORY_DIR = os.path.join(PROJECT_DIR, "history")
MODELS_DIR = os.path.join(PROJECT_DIR, "models")
CACHE_DIR = os.environ.get("PERCH_FEATURE_CACHE_DIR", os.path.join(PROJECT_DIR, "perch_feature_cache"))
OUTPUT_CSV = os.environ.get("OUTPUT_CSV", os.path.join(PROJECT_DIR, "submission.csv"))
BASELINE_RAW_CSV = os.environ.get("BASELINE_RAW_CSV", os.path.join(HISTORY_DIR, "baseline_raw_submission.csv"))

DEFAULT_BASELINE_MODEL_PATH = (
    "/kaggle/input/datasets/unsseo/perch-feature/"
    "9de231cebc7ba63a8bc4d2adb87fa3cbff50481d2b7b671582e68a7bdcf40b66.pth"
)
DEFAULT_BASELINE_HISTORY_CSV = (
    "/kaggle/input/datasets/unsseo/perch-feature/"
    "9de231cebc7ba63a8bc4d2adb87fa3cbff50481d2b7b671582e68a7bdcf40b66.csv"
)
DEFAULT_PERCH_PROBE_MODEL_PATHS = (
    "/kaggle/input/datasets/unsseo/perch-feature/perch_feature_probe_full.pt"
)
DEFAULT_PERCH_PROBE_HISTORY_CSV = (
    "/kaggle/input/datasets/unsseo/perch-feature/perch_feature_probe_history.csv"
)
DEFAULT_PERCH_MODEL_DIR = (
    "/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2/2"
)
DEFAULT_PERCH_ONNX_PATH = (
    "/kaggle/input/datasets/unsseo/perch-onnx-backbone/perch_v2_backbone.onnx"
)

os.environ.setdefault("BASELINE_MODEL_PATH", DEFAULT_BASELINE_MODEL_PATH)
os.environ.setdefault("BASELINE_HISTORY_CSV", DEFAULT_BASELINE_HISTORY_CSV)
os.environ.setdefault("PERCH_PROBE_MODEL_PATHS", DEFAULT_PERCH_PROBE_MODEL_PATHS)
os.environ.setdefault("PERCH_PROBE_HISTORY_CSV", DEFAULT_PERCH_PROBE_HISTORY_CSV)
os.environ.setdefault("PERCH_MODEL_DIR", DEFAULT_PERCH_MODEL_DIR)
os.environ.setdefault("PERCH_ONNX_PATH", DEFAULT_PERCH_ONNX_PATH)
os.environ.setdefault("PERCH_USE_ONNX", "1")

os.makedirs(HISTORY_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
os.environ.setdefault("BIRDCLEF_DATA_PATH", DATA_PATH)


def set_seed(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def first_existing_column(df, names):
    for name in names:
        if name in df.columns:
            return name
    return None


def load_taxonomy_labels(data_path):
    taxonomy_path = os.path.join(data_path, "taxonomy.csv")
    if not os.path.exists(taxonomy_path):
        raise FileNotFoundError(f"taxonomy.csv was not found under {data_path}")
    taxonomy = pd.read_csv(taxonomy_path)
    label_col = first_existing_column(taxonomy, ["primary_label", "ebird_code", "species_code", "label"])
    if label_col is None:
        raise ValueError("taxonomy.csv must contain a primary_label, ebird_code, species_code, or label column.")
    return taxonomy[label_col].dropna().astype(str).tolist()


def load_submission_labels(data_path):
    sample_path = os.path.join(data_path, "sample_submission.csv")
    if os.path.exists(sample_path):
        sample = pd.read_csv(sample_path, nrows=1)
        return [col for col in sample.columns if col != "row_id"]
    return load_taxonomy_labels(data_path)


def load_train_audio_labels(data_path):
    train = pd.read_csv(os.path.join(data_path, "train.csv"))
    if "primary_label" not in train.columns:
        return []
    return sorted(train["primary_label"].dropna().astype(str).unique().tolist())


def safe_eval_config(value):
    if not isinstance(value, str):
        return value
    text = value.strip()
    if not text:
        return value
    try:
        return ast.literal_eval(text)
    except Exception:
        pass
    try:
        return eval(text, {"__builtins__": {}}, {})
    except Exception:
        return value


def decode_config(cfg):
    return {key: safe_eval_config(value) for key, value in cfg.items()}


def env_int(name, default):
    raw = os.environ.get(name, str(default))
    token = str(raw).strip().split()[0]
    return int(token)


def env_float(name, default):
    raw = os.environ.get(name, str(default))
    token = str(raw).strip().split()[0]
    return float(token)


def format_time(t):
    h = int(t / 3600)
    t -= h * 3600
    m = int(t / 60)
    t -= m * 60
    out = ""
    if h > 0:
        out += str(h) + ":"
    return out + str(m).zfill(2) + ":" + str(int(t)).zfill(2)


def load_baseline_state(path):
    try:
        return torch.load(path, weights_only=True, map_location="cpu")
    except TypeError:
        return torch.load(path, map_location="cpu")


def load_probe_state(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def clean_state_dict(state):
    state_dict = state.get("state_dict", state) if isinstance(state, dict) else state
    if isinstance(state_dict, dict) and "model" in state_dict and isinstance(state_dict["model"], dict):
        state_dict = state_dict["model"]
    cleaned = {}
    for key, value in state_dict.items():
        if key.startswith("module."):
            key = key[len("module.") :]
        cleaned[key] = value
    return cleaned


def infer_num_labels_from_state(state):
    state_dict = clean_state_dict(state)
    for key, value in state_dict.items():
        if not hasattr(value, "shape") or len(value.shape) != 2:
            continue
        if "classifier" in key or "head" in key or key.endswith("fc.weight"):
            return int(value.shape[0])
    return None


def find_baseline_model_path():
    explicit = os.environ.get("BASELINE_MODEL_PATH")
    if explicit:
        if not os.path.exists(explicit):
            raise FileNotFoundError(f"BASELINE_MODEL_PATH does not exist: {explicit}")
        return explicit
    exp_id = os.environ.get("BASELINE_EXP_ID")
    if exp_id:
        candidates = glob.glob(f"/kaggle/input/**/{exp_id}.pth", recursive=True)
        candidates.append(os.path.join(MODELS_DIR, f"{exp_id}.pth"))
        for path in candidates:
            if os.path.exists(path):
                return path
    candidates = [
        path for path in sorted(glob.glob("/kaggle/input/**/*.pth", recursive=True), key=os.path.getmtime, reverse=True)
        if "perch_feature_probe" not in os.path.basename(path)
    ]
    candidates.extend([
        path for path in sorted(glob.glob(os.path.join(MODELS_DIR, "*.pth")), key=os.path.getmtime, reverse=True)
        if "perch_feature_probe" not in os.path.basename(path)
    ])
    if candidates:
        return candidates[0]
    raise FileNotFoundError("Set BASELINE_MODEL_PATH or attach a Kaggle dataset containing the baseline .pth file.")


def find_baseline_history_csv(model_path):
    explicit = os.environ.get("BASELINE_HISTORY_CSV")
    if explicit:
        if not os.path.exists(explicit):
            raise FileNotFoundError(f"BASELINE_HISTORY_CSV does not exist: {explicit}")
        return explicit
    exp_id = os.environ.get("BASELINE_EXP_ID") or os.path.splitext(os.path.basename(model_path))[0]
    candidates = [
        os.path.join(HISTORY_DIR, f"{exp_id}.csv"),
        *glob.glob(f"/kaggle/input/**/{exp_id}.csv", recursive=True),
        *glob.glob(f"/kaggle/input/**/history/{exp_id}.csv", recursive=True),
    ]
    for path in candidates:
        if os.path.exists(path):
            return path
    history_files = [
        path for path in sorted(glob.glob("/kaggle/input/**/*.csv", recursive=True), key=os.path.getmtime, reverse=True)
        if os.path.basename(path) not in {"sample_submission.csv", "submission.csv"}
        and "perch_feature_" not in os.path.basename(path)
        and "baseline_perch_zero_shot_hybrid" not in os.path.basename(path)
    ]
    return history_files[0] if history_files else None


class Spectrogram(nn.Module):
    def __init__(
        self,
        sr=32000,
        n_fft=2048,
        n_mels=256,
        hop_length=512,
        f_min=20,
        f_max=16000,
        channels=1,
        norm="slaney",
        mel_scale="htk",
        target_size=(256, 256),
        top_db=80.0,
        delta_win=5,
        spec_aug=None,
    ):
        super().__init__()
        self.channels = channels
        self.top_db = top_db
        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=sr,
            n_fft=n_fft,
            hop_length=hop_length,
            n_mels=n_mels,
            f_min=f_min,
            f_max=f_max,
            mel_scale=mel_scale,
            pad_mode="reflect",
            power=2.0,
            norm=norm,
            center=True,
        )
        self.resize = torchvision.transforms.Resize(size=target_size)

    def power_to_db(self, spec):
        amin = 1e-10
        log_spec = 10.0 * torch.log10(spec.clamp(min=amin))
        log_spec -= 10.0 * torch.log10(torch.tensor(amin).to(spec))
        if self.top_db is not None:
            max_val = log_spec.flatten(-2).max(dim=-1).values[..., None, None]
            log_spec = torch.maximum(log_spec, max_val - self.top_db)
        return log_spec

    def forward(self, x):
        squeeze = False
        if x.dim() == 1:
            x = x.unsqueeze(0)
            squeeze = True
        mel_spec = self.mel_transform(x)
        mel_spec = self.power_to_db(mel_spec)
        mel_spec = mel_spec.unsqueeze(1).repeat(1, self.channels, 1, 1)
        mel_spec = self.resize(mel_spec)
        flat = mel_spec.view(mel_spec.shape[0], mel_spec.shape[1], -1)
        mins = flat.min(dim=-1).values[..., None, None]
        maxs = flat.max(dim=-1).values[..., None, None]
        mel_spec = (mel_spec - mins) / (maxs - mins + 1e-7)
        if squeeze:
            mel_spec = mel_spec.squeeze(0)
        return mel_spec


class BirdModel(nn.Module):
    def __init__(self, config=None):
        super().__init__()
        import timm

        self.config = {
            "scale": 1,
            "backbone_pooling": "avg",
            "backbone": "tf_efficientnetv2_b0",
            "dropout": 0.1,
            "pretrained": False,
            "channels": 1,
            "num_labels": 234,
        }
        if config:
            self.config.update(config)

        self.backbone = timm.create_model(
            self.config["backbone"],
            pretrained=False,
            num_classes=int(self.config["num_labels"]),
            global_pool=self.config["backbone_pooling"],
            in_chans=int(self.config.get("channels", 1)),
            drop_rate=float(self.config["dropout"]),
        )

    def forward(self, x):
        return self.backbone(x)


def prediction_audio_dirs():
    dirs = []
    for key in ["HYBRID_PREDICTION_AUDIO_DIR", "TEST_AUDIO_DIR"]:
        value = os.environ.get(key)
        if value:
            dirs.append(value)
    dirs.extend([
        os.path.join(DATA_PATH, "test_soundscapes"),
        os.path.join(DATA_PATH, "train_soundscapes"),
    ])
    out = []
    seen = set()
    for path in dirs:
        if path and path not in seen:
            seen.add(path)
            out.append(path)
    return out


def list_audio_files(audio_dir):
    audio_exts = (".ogg", ".wav", ".flac", ".mp3")
    if not os.path.isdir(audio_dir):
        return []
    return [
        os.path.join(audio_dir, name)
        for name in sorted(os.listdir(audio_dir))
        if name.lower().endswith(audio_exts)
    ]


def find_prediction_audio_paths():
    explicit_dir = os.environ.get("HYBRID_PREDICTION_AUDIO_DIR") or os.environ.get("TEST_AUDIO_DIR")
    test_dir = os.path.join(DATA_PATH, "test_soundscapes")
    train_dir = os.path.join(DATA_PATH, "train_soundscapes")
    has_test_files = bool(list_audio_files(test_dir))

    for audio_dir in prediction_audio_dirs():
        paths = list_audio_files(audio_dir)
        if not paths:
            continue
        if audio_dir == train_dir and not has_test_files:
            allow_full_train = os.environ.get("ALLOW_FULL_TRAIN_SOUNDSCAPE_DEBUG", "0") == "1"
            debug_limit_raw = os.environ.get("DEBUG_TRAIN_SOUNDSCAPE_LIMIT", "16").strip().lower()
            if allow_full_train or debug_limit_raw in {"all", "none", "-1"}:
                print(
                    "No test_soundscapes found; using full train_soundscapes because "
                    "ALLOW_FULL_TRAIN_SOUNDSCAPE_DEBUG=1 or DEBUG_TRAIN_SOUNDSCAPE_LIMIT=all."
                )
            else:
                debug_limit = int(debug_limit_raw)
                paths = paths[:debug_limit]
                if explicit_dir:
                    print(
                        "No test_soundscapes found; HYBRID_PREDICTION_AUDIO_DIR/TEST_AUDIO_DIR points to "
                        f"train_soundscapes, so limiting debug files to {len(paths)}. "
                        "Set ALLOW_FULL_TRAIN_SOUNDSCAPE_DEBUG=1 to run all train_soundscapes."
                    )
                else:
                    print("No test_soundscapes found; using train_soundscapes debug files:", len(paths))
        os.environ["HYBRID_PREDICTION_AUDIO_DIR"] = audio_dir
        print("prediction audio dir:", audio_dir)
        return paths

    raise FileNotFoundError(f"No audio files found under any of: {prediction_audio_dirs()}")


def load_audio_segments(filepath, sr=32000, dur=5, total_sec=60):
    wav, input_sr = torchaudio.load(filepath)
    wav = wav.float().mean(dim=0)
    if input_sr != sr:
        wav = torchaudio.functional.resample(wav, input_sr, sr)
    total_len = int(sr * total_sec)
    if wav.numel() < total_len:
        padded = torch.zeros(total_len, dtype=wav.dtype)
        padded[: wav.numel()] = wav
        wav = padded
    else:
        wav = wav[:total_len]
    seg_len = int(sr * dur)
    n_seg = int(total_sec / dur)
    return wav.reshape(n_seg, seg_len)


def predict_baseline_file(filepath, model, spec, device, cfg):
    dur = int(cfg.get("dur", 5))
    sr = int(cfg.get("sr", 32000))
    total_sec = int(cfg.get("total_sec", 60))
    segments = load_audio_segments(filepath, sr=sr, dur=dur, total_sec=total_sec)
    stem = os.path.splitext(os.path.basename(filepath))[0]
    row_ids = [f"{stem}_{end_sec}" for end_sec in (np.arange(len(segments)) * dur + dur).astype(int)]
    model.eval()
    spec.eval()
    with torch.no_grad():
        mel = spec(segments.to(device))
        pred = torch.sigmoid(model(mel)).detach().cpu().numpy().astype("float32")
    return pred, row_ids


def build_baseline_submission():
    set_seed(int(os.environ.get("BIRDCLEF_SEED", "2")))
    device = torch.device(os.environ.get("BASELINE_INFER_DEVICE", "cuda" if torch.cuda.is_available() else "cpu"))
    submission_labels = load_submission_labels(DATA_PATH)
    train_audio_labels = load_train_audio_labels(DATA_PATH)
    model_path = find_baseline_model_path()
    history_path = find_baseline_history_csv(model_path)

    cfg = {}
    if history_path and os.path.exists(history_path):
        history = pd.read_csv(history_path)
        config_cols = history.columns[7:] if len(history.columns) > 7 else []
        cfg = decode_config({col: history.iloc[0][col] for col in config_cols})

    state = load_baseline_state(model_path)
    inferred_num_labels = infer_num_labels_from_state(state)
    if inferred_num_labels is not None:
        cfg["num_labels"] = inferred_num_labels
    cfg["pretrained"] = False
    cfg.setdefault("mel", {"n_mels": 256, "f_min": 20, "n_fft": 2048, "target_size": (256, 256)})
    cfg.setdefault("sr", 32000)
    cfg.setdefault("dur", 5)
    cfg.setdefault("total_sec", 60)

    print("DATA_PATH:", DATA_PATH)
    print("baseline model:", model_path)
    print("baseline history:", history_path)
    print("baseline output labels:", cfg.get("num_labels"))
    print("device:", device)

    spec = Spectrogram(**cfg["mel"]).to(device)
    model = BirdModel(config=cfg).to(device)
    model.load_state_dict(clean_state_dict(state), strict=True)

    paths = find_prediction_audio_paths()
    all_pred = []
    all_row_ids = []
    start_time = time()
    for idx, filepath in enumerate(paths, start=1):
        pred, row_ids = predict_baseline_file(filepath, model, spec, device, cfg)
        all_pred.append(pred)
        all_row_ids.extend(row_ids)
        if idx == 1 or idx % 16 == 0 or idx == len(paths):
            fps = idx / max(time() - start_time, 1e-6)
            remaining = (len(paths) - idx) / max(fps, 1e-6)
            print(f"baseline {idx}/{len(paths)} elapsed {format_time(time() - start_time)} remaining {format_time(remaining)}")

    pred = np.concatenate(all_pred, axis=0)
    baseline_df = pd.DataFrame(np.zeros((len(pred), len(submission_labels)), dtype=np.float32), columns=submission_labels)
    if pred.shape[1] == len(train_audio_labels):
        baseline_df.loc[:, train_audio_labels] = pred
        print("filled baseline scores for train_audio labels:", len(train_audio_labels))
    elif pred.shape[1] == len(submission_labels):
        baseline_df.loc[:, submission_labels] = pred
        print("filled baseline scores for all labels:", len(submission_labels))
    else:
        raise ValueError(
            f"Baseline model output dim {pred.shape[1]} does not match train_audio labels "
            f"({len(train_audio_labels)}) or submission labels ({len(submission_labels)})."
        )
    baseline_df.insert(0, "row_id", all_row_ids)
    baseline_df.to_csv(BASELINE_RAW_CSV, index=False)
    print("saved baseline raw submission:", BASELINE_RAW_CSV)
    return baseline_df


def parse_time_seconds(value):
    if pd.isna(value):
        return None
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)
    text = str(value).strip()
    if not text:
        return None
    try:
        return float(text)
    except ValueError:
        pass
    parts = text.split(":")
    if len(parts) in {2, 3}:
        try:
            nums = [float(part) for part in parts]
        except ValueError:
            return None
        if len(nums) == 2:
            minutes, seconds = nums
            return minutes * 60.0 + seconds
        hours, minutes, seconds = nums
        return hours * 3600.0 + minutes * 60.0 + seconds
    return None


def resolve_soundscape_audio_path(soundscape_dir, filename):
    filename = os.path.basename(str(filename))
    direct = os.path.join(soundscape_dir, filename)
    if os.path.exists(direct):
        return direct
    stem, ext = os.path.splitext(filename)
    candidates = [filename] if ext else []
    if not ext:
        for audio_ext in [".ogg", ".wav", ".flac", ".mp3"]:
            candidates.append(filename + audio_ext)
    for candidate in candidates:
        path = os.path.join(soundscape_dir, candidate)
        if os.path.exists(path):
            return path
    for path in glob.glob(os.path.join(soundscape_dir, "*")):
        if os.path.splitext(os.path.basename(path))[0] == stem:
            return path
    return None


def parse_prediction_row_id(row_id, window_sec, time_mode="end"):
    text = str(row_id)
    stem, sep, sec_text = text.rpartition("_")
    if not sep:
        return text, None
    seconds = parse_time_seconds(sec_text)
    if seconds is None:
        return text, None
    start = seconds if time_mode == "start" else max(0.0, seconds - float(window_sec))
    return stem, float(start)


def build_prediction_rows(baseline_df, window_sec=5.0):
    audio_dirs = prediction_audio_dirs()
    row_id_time_mode = os.environ.get("HYBRID_ROW_ID_TIME_MODE", "end")
    rows = []
    missing = []
    for row_pos, row in baseline_df.iterrows():
        filename, start = parse_prediction_row_id(row["row_id"], window_sec, time_mode=row_id_time_mode)
        if start is None:
            start = 0.0
        filepath = None
        for audio_dir in audio_dirs:
            filepath = resolve_soundscape_audio_path(audio_dir, filename)
            if filepath is not None:
                break
        if filepath is None:
            missing.append(filename)
            continue
        rows.append({
            "row_position": int(row_pos),
            "filename": os.path.basename(filepath),
            "filepath": filepath,
            "start_seconds": round(float(start), 3),
        })
    if missing:
        raise FileNotFoundError(
            f"Missing audio files for {len(missing)} prediction rows under {audio_dirs}, e.g. {missing[:5]}"
        )
    rows = pd.DataFrame(rows)
    print("Perch prediction rows:", len(rows), "audio_dirs:", audio_dirs)
    return rows


def row_id_file_id(row_id):
    stem, sep, sec_text = str(row_id).rpartition("_")
    if sep and parse_time_seconds(sec_text) is not None:
        return stem
    return str(row_id)


def topk_mean(values, k):
    if values.shape[1] == 0:
        return np.zeros(values.shape[0], dtype=np.float32)
    k = max(1, min(int(k), values.shape[1]))
    if k == 1:
        return values.max(axis=1)
    part = np.partition(values, -k, axis=1)[:, -k:]
    return part.mean(axis=1)


def select_perch_candidate_rows(baseline_df, zero_shot_labels, train_audio_labels):
    max_rows = env_int("PERCH_MAX_ROWS", 3000)
    top_per_file = env_int("PERCH_TOP_WINDOWS_PER_FILE", 1)
    select_by = os.environ.get("PERCH_SELECT_BY", "activity").strip().lower()
    activity_topk = env_int("PERCH_ACTIVITY_TOPK", 5)
    min_score = env_float("PERCH_MIN_ACTIVITY", 0.0)

    if max_rows == 0:
        print("PERCH_MAX_ROWS=0: applying Perch to all rows.")
        return baseline_df.copy()

    label_columns = [col for col in baseline_df.columns if col != "row_id"]
    seen_columns = [label for label in train_audio_labels if label in baseline_df.columns]
    zero_columns = [label for label in zero_shot_labels if label in baseline_df.columns]

    if select_by in {"zero", "zero_shot", "zeroshot"} and zero_columns:
        score_matrix = baseline_df[zero_columns].to_numpy(dtype=np.float32, copy=False)
        scores = topk_mean(score_matrix, activity_topk)
    elif select_by in {"all", "max_all"}:
        score_matrix = baseline_df[label_columns].to_numpy(dtype=np.float32, copy=False)
        scores = topk_mean(score_matrix, activity_topk)
    else:
        score_matrix = baseline_df[seen_columns or label_columns].to_numpy(dtype=np.float32, copy=False)
        scores = topk_mean(score_matrix, activity_topk)

    candidates = pd.DataFrame({
        "row_index": np.arange(len(baseline_df)),
        "row_id": baseline_df["row_id"].astype(str).values,
        "score": scores,
    })
    candidates["file_id"] = candidates["row_id"].map(row_id_file_id)
    if min_score > 0:
        candidates = candidates[candidates["score"] >= min_score].copy()

    if top_per_file > 0 and not candidates.empty:
        candidates = (
            candidates.sort_values(["file_id", "score"], ascending=[True, False])
            .groupby("file_id", as_index=False, sort=False)
            .head(top_per_file)
        )

    if max_rows > 0 and len(candidates) > max_rows:
        candidates = candidates.sort_values("score", ascending=False).head(max_rows)

    candidates = candidates.sort_values("row_index")
    selected = baseline_df.iloc[candidates["row_index"].to_numpy()].copy()
    print(
        "limited Perch row selection:",
        len(selected),
        "/",
        len(baseline_df),
        "select_by:",
        select_by,
        "top_per_file:",
        top_per_file,
        "max_rows:",
        max_rows,
        "min_score:",
        min_score,
    )
    if not selected.empty:
        print(
            "selection score stats:",
            "min",
            float(candidates["score"].min()),
            "median",
            float(candidates["score"].median()),
            "max",
            float(candidates["score"].max()),
        )
    return selected


def load_audio_window(filepath, start_seconds, sample_rate=32000, window_sec=5.0):
    wav, sr = torchaudio.load(filepath)
    wav = wav.float().mean(dim=0)
    if sr != sample_rate:
        wav = torchaudio.functional.resample(wav, sr, sample_rate)
    target_len = int(round(sample_rate * window_sec))
    start = max(0, int(round(float(start_seconds) * sample_rate)))
    segment = wav[start : start + target_len]
    if segment.numel() < target_len:
        padded = torch.zeros(target_len, dtype=wav.dtype)
        padded[: segment.numel()] = segment
        segment = padded
    return segment.detach().cpu().numpy().astype("float32")


def find_perch_saved_model_dir():
    explicit = os.environ.get("PERCH_MODEL_DIR")
    if explicit:
        if not os.path.exists(os.path.join(explicit, "saved_model.pb")):
            raise FileNotFoundError(f"PERCH_MODEL_DIR must contain saved_model.pb: {explicit}")
        return explicit
    candidates = glob.glob("/kaggle/input/**/saved_model.pb", recursive=True)
    if candidates:
        return os.path.dirname(candidates[0])
    return None


def configure_tensorflow():
    import tensorflow as tf
    try:
        for gpu in tf.config.list_physical_devices("GPU"):
            tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass
    return tf


def load_perch_model():
    tf = configure_tensorflow()
    model_dir = find_perch_saved_model_dir()
    if model_dir and os.path.exists(model_dir):
        print("Loading Perch SavedModel:", model_dir)
        model = tf.saved_model.load(model_dir)
        if hasattr(model, "signatures"):
            print("Perch SavedModel signatures:", list(model.signatures.keys()))
        return model
    if os.path.isdir("/kaggle/input"):
        raise FileNotFoundError(
            "Perch SavedModel was not found under /kaggle/input. "
            "Attach the Perch model dataset or set PERCH_MODEL_DIR."
        )
    import tensorflow_hub as hub
    handle = os.environ.get("PERCH_MODEL_HANDLE", "https://tfhub.dev/google/bird-vocalization-classifier/4")
    print("Loading Perch from TFHub:", handle)
    return hub.load(handle)


def find_perch_onnx_path():
    explicit = os.environ.get("PERCH_ONNX_PATH")
    if explicit:
        if not os.path.exists(explicit):
            raise FileNotFoundError(f"PERCH_ONNX_PATH does not exist: {explicit}")
        return explicit
    candidates = sorted(glob.glob("/kaggle/input/**/*.onnx", recursive=True))
    if candidates:
        return candidates[0]
    return None


def import_onnxruntime(use_onnx):
    try:
        import onnxruntime as ort
        return ort
    except ImportError as exc:
        if use_onnx != "auto":
            raise ImportError(
                "onnxruntime is required for PERCH_USE_ONNX=1. "
                "Install it in a notebook cell before running this script, or disable the zero-shot probe."
            ) from exc
        raise


def load_perch_engine():
    use_onnx = os.environ.get("PERCH_USE_ONNX", "auto").strip().lower()
    onnx_path = find_perch_onnx_path()
    if use_onnx in {"1", "true", "yes", "auto"} and onnx_path:
        try:
            ort = import_onnxruntime(use_onnx)
        except ImportError as exc:
            if use_onnx != "auto":
                raise
        else:
            providers = ["CPUExecutionProvider"]
            print("Loading Perch ONNX:", onnx_path)
            session = ort.InferenceSession(onnx_path, providers=providers)
            print("Perch ONNX inputs:", [(inp.name, inp.shape, inp.type) for inp in session.get_inputs()])
            print("Perch ONNX outputs:", [(out.name, out.shape, out.type) for out in session.get_outputs()])
            return "onnx", session
    return "tf", load_perch_model()


def tensor_to_numpy(value):
    if hasattr(value, "numpy"):
        value = value.numpy()
    return np.asarray(value)


def dict_get_first(mapping, keys):
    for key in keys:
        if key in mapping:
            return mapping[key]
    return None


def flatten_perch_output(value, batch_size):
    arr = tensor_to_numpy(value).astype("float32")
    if arr.ndim == 0:
        arr = arr.reshape(1, 1)
    if arr.ndim == 1:
        arr = arr.reshape(batch_size, -1) if arr.size % batch_size == 0 else arr.reshape(1, -1)
    if arr.ndim > 2:
        arr = arr.reshape(arr.shape[0], -1)
    return arr


def output_dim(value, batch_size):
    try:
        arr = flatten_perch_output(value, batch_size)
    except Exception:
        return -1
    if arr.shape[0] != batch_size:
        return -1
    return int(arr.shape[1]) if arr.ndim >= 2 else -1


def choose_perch_output(outputs, batch_size, role):
    if not isinstance(outputs, dict):
        return None

    if role == "embedding":
        env_key = os.environ.get("PERCH_EMBEDDING_KEY")
        preferred = ["embedding", "embeddings", "embed", "feature", "features", "embedding_output"]
    else:
        env_key = os.environ.get("PERCH_LOGITS_KEY")
        preferred = ["logits", "label", "scores", "class_logits", "classifier", "prediction", "predictions"]

    if env_key:
        if env_key not in outputs:
            raise KeyError(f"{env_key} was set for {role}, but available outputs are {list(outputs.keys())}")
        return outputs[env_key]

    found = dict_get_first(outputs, preferred)
    if found is not None:
        return found

    dims = []
    for key, value in outputs.items():
        dim = output_dim(value, batch_size)
        if dim > 0:
            dims.append((key, dim, value))
    if not dims:
        return None

    if role == "embedding":
        # Perch embeddings are usually 1280 or 1536 dims. Prefer those shapes, then
        # the smaller non-logit dense vector.
        dims = sorted(dims, key=lambda item: (min(abs(item[1] - 1280), abs(item[1] - 1536)), item[1]))
        return dims[0][2]

    # Classifier logits are usually much wider than embeddings.
    dims = sorted(dims, key=lambda item: item[1], reverse=True)
    return dims[0][2]


def call_saved_model_signature(model, audio_batch):
    import tensorflow as tf

    if not hasattr(model, "signatures") or not model.signatures:
        raise AttributeError("Loaded Perch SavedModel has neither infer_tf nor callable signatures.")

    signature_key = os.environ.get("PERCH_SIGNATURE_KEY", "serving_default")
    if signature_key not in model.signatures:
        signature_key = next(iter(model.signatures.keys()))
    fn = model.signatures[signature_key]
    audio_tensor = tf.convert_to_tensor(audio_batch.astype("float32"))

    args, kwargs = fn.structured_input_signature
    input_kwargs = {key: value for key, value in kwargs.items() if key != "unknown_options"}
    if input_kwargs:
        input_name = os.environ.get("PERCH_INPUT_KEY") or next(iter(input_kwargs.keys()))
        print("Perch signature:", signature_key, "input:", input_name)
        return fn(**{input_name: audio_tensor})

    print("Perch signature:", signature_key, "positional input")
    return fn(audio_tensor)


def split_perch_outputs(outputs, batch_size):
    if isinstance(outputs, dict):
        logits = choose_perch_output(outputs, batch_size, "logits")
        embedding = choose_perch_output(outputs, batch_size, "embedding")
    elif isinstance(outputs, (tuple, list)):
        flattened = [flatten_perch_output(value, batch_size) for value in outputs]
        valid = [(idx, arr) for idx, arr in enumerate(flattened) if arr.shape[0] == batch_size and arr.ndim >= 2]
        if not valid:
            raise ValueError("Perch output did not contain batch-aligned tensors.")
        logits_idx, logits_arr = max(valid, key=lambda item: item[1].shape[1])
        emb_idx, emb_arr = min(
            valid,
            key=lambda item: (min(abs(item[1].shape[1] - 1280), abs(item[1].shape[1] - 1536)), item[1].shape[1]),
        )
        logits = logits_arr
        embedding = emb_arr
    else:
        raise ValueError(f"Unsupported Perch output type: {type(outputs)}")

    if embedding is None:
        raise ValueError(
            "Could not identify Perch embedding output. "
            "Set PERCH_EMBEDDING_KEY to one of the SavedModel output names."
        )

    embedding = flatten_perch_output(embedding, batch_size) if not isinstance(embedding, np.ndarray) else embedding
    if logits is None:
        logits = np.zeros((batch_size, 0), dtype=np.float32)
    else:
        logits = flatten_perch_output(logits, batch_size) if not isinstance(logits, np.ndarray) else logits

    if os.environ.get("PERCH_DEBUG_OUTPUTS", "0") == "1" and isinstance(outputs, dict):
        debug_shapes = {key: tuple(tensor_to_numpy(value).shape) for key, value in outputs.items()}
        print("Perch output shapes:", debug_shapes)
        print("Selected logits dim:", logits.shape, "embedding dim:", embedding.shape)
    return logits.astype("float32"), embedding.astype("float32")


def infer_perch_batch(model, audio_batch):
    audio_batch = audio_batch.astype("float32")
    if hasattr(model, "infer_tf"):
        outputs = model.infer_tf(audio_batch)
    else:
        outputs = call_saved_model_signature(model, audio_batch)
    return split_perch_outputs(outputs, batch_size=audio_batch.shape[0])


def infer_perch_onnx_batch(session, audio_batch):
    audio_batch = audio_batch.astype("float32")
    input_name = os.environ.get("PERCH_ONNX_INPUT_KEY") or session.get_inputs()[0].name
    outputs = session.run(None, {input_name: audio_batch})
    return split_perch_outputs(outputs, batch_size=audio_batch.shape[0])


def extract_or_load_perch_features(rows, labels, feature_cfg):
    cache_path = feature_cfg["cache_path"]
    feature_mode = feature_cfg.get("feature_mode", "emb").lower()
    force = bool(feature_cfg.get("force_recreate", False))
    if os.path.exists(cache_path) and not force:
        print("Loading cached Perch prediction features:", cache_path)
        data = np.load(cache_path, allow_pickle=True)
        cached_n = int(data["emb"].shape[0])
        if cached_n != len(rows):
            print("Ignoring Perch feature cache because row count differs:", cached_n, "!=", len(rows))
        elif "row_position" in data.files and not np.array_equal(
            data["row_position"].astype(np.int64),
            rows["row_position"].to_numpy(dtype=np.int64),
        ):
            print("Ignoring Perch feature cache because selected row positions differ.")
        else:
            result = {
                "emb": data["emb"].astype("float32"),
                "logits": data["logits"].astype("float32") if "logits" in data.files else None,
            }
            if feature_mode in {"logits", "emb_logits", "concat"} and result["logits"] is None:
                raise ValueError("Feature cache does not contain logits. Recreate cache or use PERCH_FEATURE_MODE=emb.")
            return result

    engine_kind, model = load_perch_engine()
    batch_size = int(feature_cfg.get("batch_size", 16))
    sample_rate = int(feature_cfg.get("sample_rate", 32000))
    window_sec = float(feature_cfg.get("window_sec", 5.0))
    save_logits = bool(feature_cfg.get("save_logits", False)) or feature_mode in {"logits", "emb_logits", "concat"}
    embeddings = []
    logits_list = []
    t0 = time()
    n = len(rows)
    for start_idx in range(0, n, batch_size):
        batch_rows = rows.iloc[start_idx : start_idx + batch_size]
        audio = np.stack([
            load_audio_window(row.filepath, row.start_seconds, sample_rate=sample_rate, window_sec=window_sec)
            for row in batch_rows.itertuples(index=False)
        ])
        if engine_kind == "onnx":
            logits, emb = infer_perch_onnx_batch(model, audio)
        else:
            logits, emb = infer_perch_batch(model, audio)
        embeddings.append(emb)
        if save_logits:
            logits_list.append(logits)
        done = min(start_idx + batch_size, n)
        elapsed = max(1e-6, time() - t0)
        remaining = (n - done) / max(done / elapsed, 1e-6)
        print(f"perch features {done}/{n} ({done / n * 100:.1f}%) elapsed {elapsed / 60:.1f}m eta {remaining / 60:.1f}m")

    emb = np.concatenate(embeddings, axis=0).astype("float32")
    logits = np.concatenate(logits_list, axis=0).astype("float32") if save_logits else None
    save_kwargs = {
        "emb": emb,
        "labels": np.asarray(labels, dtype=object),
        "row_position": rows["row_position"].to_numpy(dtype=np.int64),
    }
    if logits is not None:
        save_kwargs["logits"] = logits
    np.savez_compressed(cache_path, **save_kwargs)
    print("Saved Perch prediction feature cache:", cache_path)
    return {"emb": emb, "logits": logits}


def build_feature_matrix(feature_data, feature_mode):
    feature_mode = feature_mode.lower()
    emb = feature_data["emb"]
    logits = feature_data.get("logits")
    if feature_mode == "emb":
        x = emb
    elif feature_mode == "logits":
        if logits is None:
            raise ValueError("PERCH_FEATURE_MODE=logits requires logits.")
        x = logits
    elif feature_mode in {"emb_logits", "concat"}:
        if logits is None:
            raise ValueError("PERCH_FEATURE_MODE=emb_logits requires logits.")
        x = np.concatenate([emb, logits], axis=1)
    else:
        raise ValueError("PERCH_FEATURE_MODE must be emb, logits, or emb_logits.")
    return np.nan_to_num(x.astype("float32"), nan=0.0, posinf=0.0, neginf=0.0)


class PerchProbe(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dims, dropout=0.2):
        super().__init__()
        layers = []
        prev = input_dim
        for dim in hidden_dims:
            layers.append(nn.Linear(prev, dim))
            layers.append(nn.ReLU(inplace=True))
            layers.append(nn.Dropout(dropout))
            prev = dim
        layers.append(nn.Linear(prev, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def parse_hidden_dims(text):
    dims = []
    for part in str(text).replace(";", ",").split(","):
        part = part.strip()
        if part:
            dims.append(int(part))
    return dims or [512, 256]


def find_probe_model_paths():
    explicit = os.environ.get("PERCH_PROBE_MODEL_PATHS")
    if explicit:
        paths = [part.strip() for part in explicit.replace(";", ",").split(",") if part.strip()]
        missing = [path for path in paths if not os.path.exists(path)]
        if missing:
            raise FileNotFoundError(f"PERCH_PROBE_MODEL_PATHS has missing files: {missing}")
        return paths
    full_models = sorted(glob.glob("/kaggle/input/**/perch_feature_probe_full.pt", recursive=True))
    if full_models:
        return [full_models[0]]
    fold_models = sorted(glob.glob("/kaggle/input/**/perch_feature_probe_fold*.pt", recursive=True))
    if fold_models:
        return fold_models
    local_full = os.path.join(MODELS_DIR, "perch_feature_probe_full.pt")
    if os.path.exists(local_full):
        return [local_full]
    return []


def find_probe_history_csv(model_paths):
    explicit = os.environ.get("PERCH_PROBE_HISTORY_CSV")
    if explicit:
        if not os.path.exists(explicit):
            raise FileNotFoundError(f"PERCH_PROBE_HISTORY_CSV does not exist: {explicit}")
        return explicit
    candidates = []
    for model_path in model_paths:
        model_dir = os.path.dirname(model_path)
        candidates.extend([
            os.path.join(model_dir, "perch_feature_probe_history.csv"),
            os.path.join(model_dir, "perch_feature_probe_manifest.json"),
        ])
    candidates.extend(glob.glob("/kaggle/input/**/perch_feature_probe_history.csv", recursive=True))
    candidates.extend(glob.glob("/kaggle/input/**/perch_feature_probe_manifest.json", recursive=True))
    for path in candidates:
        if os.path.exists(path):
            return path
    return None


def load_probe_inference_settings(model_paths):
    settings = {}
    history_path = find_probe_history_csv(model_paths)
    if history_path:
        print("probe history/manifest:", history_path)
        if history_path.endswith(".json"):
            with open(history_path, "r", encoding="utf-8") as f:
                row = json.load(f)
        else:
            row = pd.read_csv(history_path).iloc[0].to_dict()
        for key in ["feature_mode", "window_sec", "sample_rate"]:
            if key in row and not pd.isna(row[key]):
                settings[key] = row[key]
    if model_paths:
        state = load_probe_state(model_paths[0])
        cfg = state.get("cfg", {}) if isinstance(state, dict) else {}
        if "feature_mode" in cfg and "feature_mode" not in settings:
            settings["feature_mode"] = cfg["feature_mode"]
    return settings


def predict_with_probe_models(x, model_paths):
    if not model_paths:
        raise ValueError("No Perch probe model paths were provided.")
    preds = []
    labels = None
    zero_shot_labels = None
    for model_path in model_paths:
        state = load_probe_state(model_path)
        state_labels = [str(label) for label in state["labels"]]
        labels = state_labels if labels is None else labels
        if state_labels != labels:
            raise ValueError("Probe model label orders differ across models.")
        zero_shot_labels = [str(label) for label in state.get("zero_shot_labels", [])]
        cfg = state.get("cfg", {})
        device = torch.device(os.environ.get("PERCH_PROBE_INFER_DEVICE", "cuda" if torch.cuda.is_available() else "cpu"))
        mean = np.asarray(state["mean"], dtype=np.float32)
        std = np.asarray(state["std"], dtype=np.float32)
        std = np.where(std < 1e-6, 1.0, std)
        x_norm = (x - mean) / std
        model = PerchProbe(
            input_dim=x.shape[1],
            output_dim=len(labels),
            hidden_dims=parse_hidden_dims(cfg.get("hidden_dims", os.environ.get("PERCH_PROBE_HIDDEN_DIMS", "512,256"))),
            dropout=float(cfg.get("dropout", 0.0)),
        ).to(device)
        model.load_state_dict(state["model"])
        model.eval()
        fold_preds = []
        batch_size = int(os.environ.get("PERCH_PROBE_INFER_BATCH_SIZE", "512"))
        with torch.no_grad():
            for start_idx in range(0, len(x_norm), batch_size):
                xb = torch.tensor(x_norm[start_idx : start_idx + batch_size], dtype=torch.float32, device=device)
                fold_preds.append(torch.sigmoid(model(xb)).detach().cpu().numpy().astype("float32"))
        preds.append(np.concatenate(fold_preds, axis=0))
        print("loaded probe model:", model_path)
    return np.mean(np.stack(preds, axis=0), axis=0).astype("float32"), labels, zero_shot_labels


def apply_zero_shot_probe(baseline_df):
    model_paths = find_probe_model_paths()
    if not model_paths:
        raise FileNotFoundError("No Perch probe model found. Attach perch_feature_probe_full.pt or set PERCH_PROBE_MODEL_PATHS.")

    probe_settings = load_probe_inference_settings(model_paths)
    state = load_probe_state(model_paths[0])
    probe_labels = [str(label) for label in state["labels"]]
    taxonomy_labels = load_taxonomy_labels(DATA_PATH)
    train_audio_labels = set(load_train_audio_labels(DATA_PATH))
    zero_shot_labels = [label for label in taxonomy_labels if label not in train_audio_labels]

    feature_cfg = {
        "cache_path": os.environ.get(
            "PERCH_PREDICTION_FEATURE_CACHE_PATH",
            os.path.join(CACHE_DIR, "submission_perch_prediction_features.npz"),
        ),
        "force_recreate": os.environ.get("PERCH_PREDICTION_FEATURE_FORCE_RECREATE", "0") == "1",
        "feature_mode": os.environ.get("PERCH_FEATURE_MODE", str(probe_settings.get("feature_mode", "emb"))),
        "save_logits": os.environ.get("PERCH_FEATURE_SAVE_LOGITS", "0") == "1",
        "batch_size": int(os.environ.get("PERCH_FEATURE_BATCH_SIZE", "16")),
        "sample_rate": int(float(os.environ.get("PERCH_SAMPLE_RATE", probe_settings.get("sample_rate", 32000)))),
        "window_sec": float(os.environ.get("SOUNDSCAPE_WEAK_WINDOW_SEC", probe_settings.get("window_sec", 5.0))),
    }
    print("probe feature settings:", feature_cfg)
    selected_df = select_perch_candidate_rows(baseline_df, zero_shot_labels, train_audio_labels)
    if selected_df.empty:
        print("No rows selected for limited Perch; returning baseline submission.")
        return baseline_df

    pred_rows = build_prediction_rows(selected_df, window_sec=feature_cfg["window_sec"])
    try:
        feature_data = extract_or_load_perch_features(pred_rows, probe_labels, feature_cfg)
    except Exception as exc:
        if os.environ.get("PERCH_REQUIRED", "0") == "1":
            raise
        print("Perch feature extraction failed; falling back to baseline-only submission.")
        print("Perch error:", repr(exc))
        print(
            "If you want hybrid on Kaggle CPU, attach a compatible Perch ONNX model and set PERCH_ONNX_PATH, "
            "or use a TensorFlow build compatible with this SavedModel."
        )
        return baseline_df
    x_pred = build_feature_matrix(feature_data, feature_cfg["feature_mode"])
    probe_pred, probe_labels, state_zero_shot_labels = predict_with_probe_models(x_pred, model_paths)
    if state_zero_shot_labels:
        zero_shot_labels = [label for label in zero_shot_labels if label in set(state_zero_shot_labels)]

    probe_df = pd.DataFrame(probe_pred, columns=probe_labels)
    replace_labels = [label for label in zero_shot_labels if label in baseline_df.columns and label in probe_df.columns]
    if not replace_labels:
        raise ValueError("No zero-shot labels were present in both baseline submission and probe output.")
    alpha = float(os.environ.get("HYBRID_ZERO_SHOT_ALPHA", "1.0"))
    out = baseline_df.copy()
    row_positions = pred_rows["row_position"].to_numpy(dtype=np.int64)
    for label in replace_labels:
        base_values = out.iloc[row_positions][label].astype(float).to_numpy()
        probe_values = probe_df[label].astype(float).to_numpy()
        out.loc[out.index[row_positions], label] = (1.0 - alpha) * base_values + alpha * probe_values
    print("replaced zero-shot labels:", len(replace_labels), "rows:", len(row_positions), "alpha:", alpha)
    return out


def main():
    baseline_df = build_baseline_submission()
    if os.environ.get("DISABLE_ZERO_SHOT_PROBE", "0") == "1":
        final_df = baseline_df
        print("zero-shot probe disabled; saving baseline-only submission.")
    else:
        final_df = apply_zero_shot_probe(baseline_df)
    final_df.to_csv(OUTPUT_CSV, index=False)
    print("final submission:", OUTPUT_CSV)
    return OUTPUT_CSV


if __name__ == "__main__":
    main()


DATA_PATH: /kaggle/input/competitions/birdclef-2026
baseline model: /kaggle/input/notebooks/antoinemasq/birdclef-2026-pytorch-baseline-training/models/8658fa9ee3393ad66210d15d4fd2063c41d50697e9fe201dbe9153e375ae2abf.pth
baseline history: /kaggle/input/notebooks/antoinemasq/birdclef-2026-pytorch-baseline-training/history/8658fa9ee3393ad66210d15d4fd2063c41d50697e9fe201dbe9153e375ae2abf.csv
baseline output labels: 234
device: cpu
No test_soundscapes found; using train_soundscapes debug files: 16
prediction audio dir: /kaggle/input/competitions/birdclef-2026/train_soundscapes
baseline 1/16 elapsed 00:02 remaining 00:36
baseline 16/16 elapsed 00:11 remaining 00:00
filled baseline scores for all labels: 234
saved baseline raw submission: /kaggle/working/history/baseline_raw_submission.csv
probe history/manifest: /kaggle/input/datasets/unsseo/perch-onnx/perch_feature_probe_history (1).csv
probe feature settings: {'cache_path': '/kaggle/working/perch_feature_cache/submission_perch_prediction_f

In [4]:
import ast
import glob
import os

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score


DATA_PATH = "/kaggle/input/competitions/birdclef-2026"
BASELINE_CSV = "/kaggle/working/history/baseline_raw_submission.csv"
HYBRID_CSV = "/kaggle/working/submission.csv"
OUT_CSV = "/kaggle/working/history/zero_shot_metric_comparison.csv"

WINDOW_SEC = 5.0


def first_existing_column(df, names):
    for name in names:
        if name in df.columns:
            return name
    return None


def parse_label_list(value):
    if pd.isna(value):
        return []
    if isinstance(value, (list, tuple, set)):
        items = list(value)
    else:
        text = str(value).strip()
        if not text:
            return []
        items = None
        if text[0] in "[({":
            try:
                parsed = ast.literal_eval(text)
                items = list(parsed) if isinstance(parsed, (list, tuple, set)) else [parsed]
            except Exception:
                items = None
        if items is None:
            items = text.replace(",", " ").replace(";", " ").replace("|", " ").split()
    return [
        str(x).strip().strip("'\"[](){}")
        for x in items
        if str(x).strip().lower() not in {"", "nan", "none", "null", "nocall", "no_call", "background"}
    ]


def parse_time_seconds(value):
    if pd.isna(value):
        return None
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)
    text = str(value).strip()
    if not text:
        return None
    try:
        return float(text)
    except ValueError:
        pass
    parts = text.split(":")
    if len(parts) == 2:
        m, s = map(float, parts)
        return m * 60 + s
    if len(parts) == 3:
        h, m, s = map(float, parts)
        return h * 3600 + m * 60 + s
    return None


def row_id_to_file_start(row_id, window_sec=5.0):
    stem, sep, sec_text = str(row_id).rpartition("_")
    if not sep:
        return str(row_id), 0.0
    end_sec = parse_time_seconds(sec_text)
    if end_sec is None:
        return str(row_id), 0.0
    return stem, max(0.0, float(end_sec) - window_sec)


def find_soundscape_label_csv(data_path):
    candidates = [
        "train_soundscape_labels.csv",
        "train_soundscapes_labels.csv",
        "train_soundscape.csv",
        "train_soundscapes.csv",
        "soundscape_labels.csv",
        "soundscapes_labels.csv",
    ]
    for name in candidates:
        path = os.path.join(data_path, name)
        if os.path.exists(path):
            return path
    matches = sorted(glob.glob(os.path.join(data_path, "*soundscape*label*.csv")))
    if matches:
        return matches[0]
    raise FileNotFoundError("train_soundscapes label csv not found")


def build_targets(pred_df, labels):
    label_csv = find_soundscape_label_csv(DATA_PATH)
    weak = pd.read_csv(label_csv)

    file_col = first_existing_column(
        weak,
        ["filename", "file_name", "soundscape_filename", "soundscape", "audio_id", "row_id"],
    )
    label_col = first_existing_column(
        weak,
        ["primary_label", "primary_labels", "labels", "birds", "ebird_code", "species_code", "label"],
    )
    start_col = first_existing_column(
        weak,
        ["start_seconds", "start_second", "start_sec", "start", "start_time", "begin_time"],
    )
    end_col = first_existing_column(
        weak,
        ["end_seconds", "end_second", "end_sec", "end", "end_time", "seconds"],
    )

    if file_col is None or label_col is None:
        raise ValueError(f"Cannot infer filename/label columns from {label_csv}: {weak.columns.tolist()}")

    label_to_idx = {label: i for i, label in enumerate(labels)}

    pred_info = []
    for row_id in pred_df["row_id"].astype(str):
        stem, start = row_id_to_file_start(row_id, WINDOW_SEC)
        pred_info.append((stem, start, start + WINDOW_SEC))

    y = np.zeros((len(pred_df), len(labels)), dtype=np.float32)

    for _, row in weak.iterrows():
        raw_file = os.path.basename(str(row[file_col]))
        file_stem = os.path.splitext(raw_file)[0]

        row_labels = parse_label_list(row[label_col])
        row_labels = [label for label in row_labels if label in label_to_idx]
        if not row_labels:
            continue

        label_start = parse_time_seconds(row[start_col]) if start_col else None
        label_end = parse_time_seconds(row[end_col]) if end_col else None

        if label_start is None and label_end is None:
            label_start, label_end = 0.0, float("inf")
        elif label_start is None:
            label_start = max(0.0, float(label_end) - WINDOW_SEC)
        elif label_end is None:
            label_end = float(label_start) + WINDOW_SEC

        for i, (pred_stem, pred_start, pred_end) in enumerate(pred_info):
            if pred_stem != file_stem:
                continue
            overlap = max(0.0, min(pred_end, label_end) - max(pred_start, label_start))
            if overlap <= 0:
                continue
            for label in row_labels:
                y[i, label_to_idx[label]] = 1.0

    return y, label_csv


def metric_block(name, y_true, y_score, labels, selected_labels):
    rows = []
    aucs = []

    for label in selected_labels:
        if label not in labels:
            continue
        idx = labels.index(label)
        target = y_true[:, idx]
        score = y_score[:, idx]

        positives = int(target.sum())
        negatives = int(len(target) - positives)
        if positives == 0 or negatives == 0:
            auc = np.nan
        else:
            auc = roc_auc_score(target, score)
            aucs.append(auc)

        pos_mean = float(score[target == 1].mean()) if positives > 0 else np.nan
        neg_mean = float(score[target == 0].mean()) if negatives > 0 else np.nan

        rows.append({
            "group": name,
            "label": label,
            "auc": auc,
            "positives": positives,
            "negatives": negatives,
            "positive_mean": pos_mean,
            "negative_mean": neg_mean,
            "separation": pos_mean - neg_mean if positives > 0 and negatives > 0 else np.nan,
        })

    summary = {
        "group": name,
        "label": "__macro__",
        "auc": float(np.mean(aucs)) if aucs else np.nan,
        "positives": int(y_true[:, [labels.index(l) for l in selected_labels if l in labels]].sum()) if selected_labels else 0,
        "negatives": np.nan,
        "positive_mean": np.nan,
        "negative_mean": np.nan,
        "separation": np.nan,
    }
    return [summary] + rows


baseline = pd.read_csv(BASELINE_CSV)
hybrid = pd.read_csv(HYBRID_CSV)

labels = [c for c in hybrid.columns if c != "row_id"]
taxonomy = pd.read_csv(os.path.join(DATA_PATH, "taxonomy.csv"))
taxonomy_col = first_existing_column(taxonomy, ["primary_label", "ebird_code", "species_code", "label"])
taxonomy_labels = taxonomy[taxonomy_col].astype(str).tolist()

train = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))
train_audio_labels = set(train["primary_label"].astype(str).unique())

zero_shot_labels = [label for label in taxonomy_labels if label not in train_audio_labels and label in labels]
seen_labels = [label for label in labels if label in train_audio_labels]

y_true, label_csv = build_targets(hybrid, labels)
baseline_score = baseline[labels].to_numpy(dtype=np.float32)
hybrid_score = hybrid[labels].to_numpy(dtype=np.float32)

rows = []
rows += metric_block("baseline_zero_shot", y_true, baseline_score, labels, zero_shot_labels)
rows += metric_block("hybrid_zero_shot", y_true, hybrid_score, labels, zero_shot_labels)
rows += metric_block("baseline_seen", y_true, baseline_score, labels, seen_labels)
rows += metric_block("hybrid_seen", y_true, hybrid_score, labels, seen_labels)

result = pd.DataFrame(rows)
os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)
result.to_csv(OUT_CSV, index=False)

print("label csv:", label_csv)
print("zero-shot labels:", len(zero_shot_labels))
print(result[result["label"] == "__macro__"])
print("saved:", OUT_CSV)

label csv: /kaggle/input/competitions/birdclef-2026/train_soundscapes_labels.csv
zero-shot labels: 28
                  group      label       auc  positives  negatives  \
0    baseline_zero_shot  __macro__  0.659488        356        NaN   
29     hybrid_zero_shot  __macro__  0.996623        356        NaN   
58        baseline_seen  __macro__  0.788410        124        NaN   
265         hybrid_seen  __macro__  0.788410        124        NaN   

     positive_mean  negative_mean  separation  
0              NaN            NaN         NaN  
29             NaN            NaN         NaN  
58             NaN            NaN         NaN  
265            NaN            NaN         NaN  
saved: /kaggle/working/history/zero_shot_metric_comparison.csv


In [5]:
import ast, glob, os
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

DATA_PATH = "/kaggle/input/competitions/birdclef-2026"
BASELINE_CSV = "/kaggle/working/history/baseline_raw_submission.csv"
HYBRID_CSV = "/kaggle/working/submission.csv"
OUT_CSV = "/kaggle/working/history/zero_shot_metric_comparison.csv"
WINDOW_SEC = 5.0

def first_col(df, names):
    return next((c for c in names if c in df.columns), None)

def parse_labels(v):
    if pd.isna(v): return []
    s = str(v).strip()
    if not s: return []
    if s[0] in "[({":
        try:
            x = ast.literal_eval(s)
            items = list(x) if isinstance(x, (list, tuple, set)) else [x]
        except Exception:
            items = s.replace(",", " ").replace(";", " ").replace("|", " ").split()
    else:
        items = s.replace(",", " ").replace(";", " ").replace("|", " ").split()
    return [str(x).strip().strip("'\"[](){}") for x in items if str(x).strip().lower() not in {"", "nan", "none", "null", "nocall", "no_call"}]

def parse_sec(v):
    if pd.isna(v): return None
    if isinstance(v, (int, float, np.integer, np.floating)): return float(v)
    s = str(v).strip()
    try: return float(s)
    except Exception: pass
    p = s.split(":")
    try:
        if len(p) == 2: return float(p[0]) * 60 + float(p[1])
        if len(p) == 3: return float(p[0]) * 3600 + float(p[1]) * 60 + float(p[2])
    except Exception:
        return None
    return None

def row_to_file_start(row_id):
    stem, sep, sec = str(row_id).rpartition("_")
    end = parse_sec(sec) if sep else None
    return (stem, max(0.0, end - WINDOW_SEC)) if end is not None else (str(row_id), 0.0)

def find_soundscape_label_csv():
    names = [
        "train_soundscape_labels.csv", "train_soundscapes_labels.csv",
        "train_soundscape.csv", "train_soundscapes.csv",
        "soundscape_labels.csv", "soundscapes_labels.csv",
    ]
    for n in names:
        p = os.path.join(DATA_PATH, n)
        if os.path.exists(p): return p
    m = sorted(glob.glob(os.path.join(DATA_PATH, "*soundscape*label*.csv")))
    if m: return m[0]
    raise FileNotFoundError("soundscape label csv not found")

baseline = pd.read_csv(BASELINE_CSV)
hybrid = pd.read_csv(HYBRID_CSV)
labels = [c for c in hybrid.columns if c != "row_id"]

taxonomy = pd.read_csv(os.path.join(DATA_PATH, "taxonomy.csv"))
tax_col = first_col(taxonomy, ["primary_label", "ebird_code", "species_code", "label"])
taxonomy_labels = taxonomy[tax_col].astype(str).tolist()

train = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))
train_audio_labels = set(train["primary_label"].astype(str).unique())
zero_shot_labels = [l for l in taxonomy_labels if l not in train_audio_labels and l in labels]
seen_labels = [l for l in labels if l in train_audio_labels]

weak_path = find_soundscape_label_csv()
weak = pd.read_csv(weak_path)

file_col = first_col(weak, ["filename", "file_name", "soundscape_filename", "soundscape", "audio_id", "row_id"])
label_col = first_col(weak, ["primary_label", "primary_labels", "labels", "birds", "ebird_code", "species_code", "label"])
start_col = first_col(weak, ["start_seconds", "start_second", "start_sec", "start", "start_time", "begin_time"])
end_col = first_col(weak, ["end_seconds", "end_second", "end_sec", "end", "end_time", "seconds"])

label_to_idx = {l: i for i, l in enumerate(labels)}
pred_info = []
for rid in hybrid["row_id"].astype(str):
    stem, st = row_to_file_start(rid)
    pred_info.append((stem, st, st + WINDOW_SEC))

y = np.zeros((len(hybrid), len(labels)), dtype=np.float32)

for _, r in weak.iterrows():
    stem = os.path.splitext(os.path.basename(str(r[file_col])))[0]
    labs = [l for l in parse_labels(r[label_col]) if l in label_to_idx]
    if not labs: continue

    st = parse_sec(r[start_col]) if start_col else None
    en = parse_sec(r[end_col]) if end_col else None
    if st is None and en is None:
        st, en = 0.0, float("inf")
    elif st is None:
        st = max(0.0, en - WINDOW_SEC)
    elif en is None:
        en = st + WINDOW_SEC

    for i, (pstem, pst, pen) in enumerate(pred_info):
        if pstem != stem: continue
        if max(0.0, min(pen, en) - max(pst, st)) <= 0: continue
        for l in labs:
            y[i, label_to_idx[l]] = 1.0

def summarize(name, score, target_labels):
    rows, aucs = [], []
    for l in target_labels:
        if l not in label_to_idx: continue
        i = label_to_idx[l]
        yt, ys = y[:, i], score[:, i]
        pos, neg = int(yt.sum()), int(len(yt) - yt.sum())
        auc = np.nan
        if pos > 0 and neg > 0:
            auc = roc_auc_score(yt, ys)
            aucs.append(auc)
        rows.append({
            "group": name, "label": l, "auc": auc,
            "positives": pos, "negatives": neg,
            "positive_mean": float(ys[yt == 1].mean()) if pos else np.nan,
            "negative_mean": float(ys[yt == 0].mean()) if neg else np.nan,
            "separation": (float(ys[yt == 1].mean()) - float(ys[yt == 0].mean())) if pos and neg else np.nan,
        })
    rows.insert(0, {
        "group": name, "label": "__macro__",
        "auc": float(np.mean(aucs)) if aucs else np.nan,
        "positives": int(y[:, [label_to_idx[l] for l in target_labels if l in label_to_idx]].sum()) if target_labels else 0,
        "negatives": np.nan, "positive_mean": np.nan, "negative_mean": np.nan, "separation": np.nan,
    })
    return rows

base_score = baseline[labels].to_numpy(np.float32)
hyb_score = hybrid[labels].to_numpy(np.float32)

rows = []
rows += summarize("baseline_zero_shot", base_score, zero_shot_labels)
rows += summarize("hybrid_zero_shot", hyb_score, zero_shot_labels)
rows += summarize("baseline_seen", base_score, seen_labels)
rows += summarize("hybrid_seen", hyb_score, seen_labels)

result = pd.DataFrame(rows)
os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)
result.to_csv(OUT_CSV, index=False)

print("weak label csv:", weak_path)
print("zero-shot labels:", len(zero_shot_labels))
display(result[result["label"] == "__macro__"])
print("saved:", OUT_CSV)

weak label csv: /kaggle/input/competitions/birdclef-2026/train_soundscapes_labels.csv
zero-shot labels: 28


,group,label,auc,positives,negatives,positive_mean,negative_mean,separation
0,baseline_zero_shot,__macro__,0.659488,356,NaN,NaN,NaN,NaN
29,hybrid_zero_shot,__macro__,0.996623,356,NaN,NaN,NaN,NaN
58,baseline_seen,__macro__,0.788410,124,NaN,NaN,NaN,NaN
265,hybrid_seen,__macro__,0.788410,124,NaN,NaN,NaN,NaN


saved: /kaggle/working/history/zero_shot_metric_comparison.csv


In [6]:
import pandas as pd
import numpy as np

RESULT_CSV = "/kaggle/working/history/zero_shot_metric_comparison.csv"

df = pd.read_csv(RESULT_CSV)

macro = (
    df[df["label"] == "__macro__"][["group", "auc", "positives"]]
    .set_index("group")
    .sort_index()
)

required_groups = [
    "baseline_zero_shot",
    "hybrid_zero_shot",
    "baseline_seen",
    "hybrid_seen",
]
missing = [g for g in required_groups if g not in macro.index]
if missing:
    raise ValueError(f"Missing macro rows for groups: {missing}")

zero_base = float(macro.loc["baseline_zero_shot", "auc"])
zero_hyb = float(macro.loc["hybrid_zero_shot", "auc"])
seen_base = float(macro.loc["baseline_seen", "auc"])
seen_hyb = float(macro.loc["hybrid_seen", "auc"])

summary = pd.DataFrame(
    {
        "baseline_auc": [zero_base, seen_base],
        "hybrid_auc": [zero_hyb, seen_hyb],
    },
    index=["zero_shot", "seen"],
)
summary["delta"] = summary["hybrid_auc"] - summary["baseline_auc"]

zero_base_labels = df[(df["group"] == "baseline_zero_shot") & (df["label"] != "__macro__")].copy()
zero_hyb_labels = df[(df["group"] == "hybrid_zero_shot") & (df["label"] != "__macro__")].copy()

label_cmp = (
    zero_base_labels[["label", "auc", "positives"]]
    .rename(columns={"auc": "baseline_auc", "positives": "positives"})
    .merge(
        zero_hyb_labels[["label", "auc"]].rename(columns={"auc": "hybrid_auc"}),
        on="label",
        how="inner",
    )
)

label_cmp = label_cmp[label_cmp["positives"].fillna(0) > 0].copy()
label_cmp["delta"] = label_cmp["hybrid_auc"] - label_cmp["baseline_auc"]
label_cmp = label_cmp.sort_values("delta", ascending=False)

valid_zero_labels = int(label_cmp["baseline_auc"].notna().sum())
improved_zero_labels = int((label_cmp["delta"] > 0).sum())
same_zero_labels = int((label_cmp["delta"].abs() < 1e-12).sum())
worse_zero_labels = int((label_cmp["delta"] < 0).sum())

print("=== Macro AUC Summary ===")
display(summary.round(6))

print(f"valid zero-shot labels: {valid_zero_labels}")
print(f"improved zero-shot labels: {improved_zero_labels}")
print(f"same zero-shot labels: {same_zero_labels}")
print(f"worse zero-shot labels: {worse_zero_labels}")
print()

if summary.loc["zero_shot", "delta"] > 0:
    print(
        f"[판단] zero-shot 성능은 개선된 것으로 보입니다. "
        f"(macro AUC {zero_base:.6f} -> {zero_hyb:.6f}, delta {summary.loc['zero_shot', 'delta']:.6f})"
    )
else:
    print(
        f"[판단] zero-shot 성능 개선을 확인하기 어렵습니다. "
        f"(macro AUC {zero_base:.6f} -> {zero_hyb:.6f}, delta {summary.loc['zero_shot', 'delta']:.6f})"
    )

if abs(summary.loc["seen", "delta"]) < 1e-12:
    print(
        f"[보조해석] seen 성능은 사실상 동일합니다. "
        f"({seen_base:.6f} -> {seen_hyb:.6f})"
    )
else:
    print(
        f"[보조해석] seen 성능 변화도 함께 있습니다. "
        f"({seen_base:.6f} -> {seen_hyb:.6f}, delta {summary.loc['seen', 'delta']:.6f})"
    )

print("\n=== Top 10 Improved Zero-shot Labels ===")
display(label_cmp.head(10)[["label", "positives", "baseline_auc", "hybrid_auc", "delta"]].round(6))


=== Macro AUC Summary ===


,baseline_auc,hybrid_auc,delta
zero_shot,0.659488,0.996623,0.337135
seen,0.788410,0.788410,0.000000


valid zero-shot labels: 19
improved zero-shot labels: 19
same zero-shot labels: 0
worse zero-shot labels: 0

[판단] zero-shot 성능은 개선된 것으로 보입니다. (macro AUC 0.659488 -> 0.996623, delta 0.337135)
[보조해석] seen 성능은 사실상 동일합니다. (0.788410 -> 0.788410)

=== Top 10 Improved Zero-shot Labels ===


,label,positives,baseline_auc,hybrid_auc,delta
27,517063,35,0.281711,0.977252,0.695541
0,1491113,26,0.338044,0.997451,0.659407
20,47158son19,5,0.546524,0.987166,0.440642
6,47158son05,1,0.633508,1.000000,0.366492
17,47158son16,12,0.636111,0.999537,0.363426
9,47158son08,12,0.654167,1.000000,0.345833
23,47158son22,24,0.661210,1.000000,0.338790
11,47158son10,8,0.663043,0.993207,0.330163
18,47158son17,43,0.690807,0.996098,0.305291
12,47158son11,12,0.696759,1.000000,0.303241
